# 05. Quantitative Evaluation & Comparative Benchmarking
### Project: Image Change Detection using Digital Image Processing for Environmental Monitoring

This notebook demonstrates:
1. Ground truth verification metrics:
   - **Precision**: $\frac{TP}{TP + FP}$
   - **Recall**: $\frac{TP}{TP + FN}$
   - **F1-Score**: $2 \cdot \frac{P \cdot R}{P + R}$
   - **IoU** (Intersection over Union / Jaccard Index): $\frac{TP}{TP + FP + FN}$
   - **Overall Accuracy**: $\frac{TP + TN}{Total}$
2. Multi-method benchmark comparison table.
3. Comparative visualization bar charts.
4. Optional **Random Forest Machine Learning baseline** for pixel classification.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / "backend"))

from app.services.utils import load_image, generate_synthetic_pair
from app.services.preprocessing import preprocess_pair
from app.services.evaluation import compute_metrics, compare_methods, random_forest_baseline


### Step 1: Load Image Pair & Ground Truth Mask


In [ ]:
project_root = Path.cwd().parent
pair_paths = generate_synthetic_pair("deforestation", size=512, base_dir=str(project_root))

before_info = load_image(pair_paths["before_path"])
after_info = load_image(pair_paths["after_path"])
gt_info = load_image(pair_paths["ground_truth_path"])

clean_before, clean_after, _ = preprocess_pair(
    before_info["bands"], after_info["bands"], do_denoise=True, do_match=True
)

gt_mask = (gt_info["bands"][0] > 0).astype(int)
print(f"Ground Truth Change Pixels: {np.count_nonzero(gt_mask)} ({np.mean(gt_mask)*100:.2f}% of scene)")


### Step 2: Multi-Method DIP Benchmark Comparison


In [ ]:
comparison = compare_methods(
    before_bands=clean_before,
    after_bands=clean_after,
    use_case="deforestation",
    is_multispectral=True,
    ground_truth=gt_mask,
    threshold_mode="otsu",
    min_area=30
)

print(f"Best Performing DIP Method: {comparison['best_method']}")
print("
" + "=" * 75)
print(f"{'Method':<22} | {'Precision':<9} | {'Recall':<9} | {'F1-Score':<9} | {'IoU':<9} | {'Accuracy':<9}")
print("=" * 75)
for row in comparison["metrics_table"]:
    print(f"{row['method']:<22} | {row['precision']:<9.4f} | {row['recall']:<9.4f} | {row['f1_score']:<9.4f} | {row['iou']:<9.4f} | {row['accuracy']:<9.4f}")
print("=" * 75)


### Step 3: Comparative Performance Bar Chart


In [ ]:
metrics_table = comparison["metrics_table"]
method_names = [m["method"].replace("_", " ").title() for m in metrics_table]
f1_scores = [m["f1_score"] for m in metrics_table]
ious = [m["iou"] for m in metrics_table]
recalls = [m["recall"] for m in metrics_table]
precisions = [m["precision"] for m in metrics_table]

x = np.arange(len(method_names))
width = 0.2

plt.figure(figsize=(13, 6))
plt.bar(x - 1.5 * width, precisions, width, label="Precision", color="#3b82f6")
plt.bar(x - 0.5 * width, recalls, width, label="Recall", color="#10b981")
plt.bar(x + 0.5 * width, f1_scores, width, label="F1-Score", color="#f59e0b")
plt.bar(x + 1.5 * width, ious, width, label="IoU", color="#ef4444")

plt.xticks(x, method_names, fontsize=11, fontweight="bold")
plt.ylabel("Score [0.0 - 1.0]", fontsize=12)
plt.title("Comparative Performance across DIP Change Detection Methods", fontsize=14, fontweight="bold")
plt.legend(frameon=True, fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.ylim(0.0, 1.05)
plt.tight_layout()
plt.show()


### Step 4: Optional Machine Learning Benchmark (Random Forest)
A Random Forest pixel classifier trained on spectral differences, spectral index shift, SSIM, and spatial Laplacian gradients serves as an ML baseline against classical DIP techniques.


In [ ]:
rf_result = random_forest_baseline(
    before_bands=clean_before,
    after_bands=clean_after,
    ground_truth=gt_mask,
    use_case="deforestation",
    train_ratio=0.25
)

print("=" * 45)
print("RANDOM FOREST ML BASELINE METRICS")
print("=" * 45)
for k, v in rf_result["metrics"].items():
    if isinstance(v, float):
        print(f"{k:<15}: {v:.4f}")

# Plot feature importances
importances = rf_result["feature_importances"]
plt.figure(figsize=(10, 4))
plt.barh(list(importances.keys()), list(importances.values()), color="#8b5cf6")
plt.xlabel("Feature Importance", fontsize=11)
plt.title("Random Forest Pixel Feature Contribution", fontsize=13, fontweight="bold")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


### Step 5: Visual Detection Comparison (Ground Truth vs DIP vs ML)


In [ ]:
best_dip_mask = comparison["results"][comparison["best_method"]]["mask"]
rf_mask = rf_result["mask"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(gt_mask, cmap="gray")
axes[0].set_title("Ground Truth Mask", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(best_dip_mask, cmap="magma")
axes[1].set_title(f"Best DIP: {comparison['best_method']} (F1: {max(f1_scores):.4f})", fontsize=12, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(rf_mask, cmap="magma")
axes[2].set_title(f"Random Forest ML (F1: {rf_result['metrics']['f1_score']:.4f})", fontsize=12, fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()
